# Running Light Curves

In [33]:
# --- In your notebook ---
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd
from itertools import islice
from IPython.display import display
import re
#imports
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import importlib
import sys
import os
from astropy.cosmology import Planck18 as cosmo
import ast



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:

repo_root = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# sanity checks
assert (repo_root / "py_files").is_dir(), "py_files/ folder not found"
assert (repo_root / "py_files" / "export_slsne_photometry.py").is_file(), "module file not found"

from py_files.export_slsne_photometry import (
    read_supernova_table_txt, to_export,
    process_one, process_all_events, load_allparams_robust, print_cols, resolve_cols 
)

supernovae_dir = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/supernovae")
out_perevent   = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files")
out_allevent   = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events")


ModuleNotFoundError: No module named 'slsne'

## Parameter Control, Test directory locations, and presence of specific event can be found. 

In [29]:
# Output format controls
write_parquet = True   # set False if you don't want parquet
write_csv = True   # CSV mirror for auditing

# Quick sanity checks
print("Repo root:        ", repo_root)
print("Supernovae dir:   ", supernovae_dir, "exists:", supernovae_dir.exists())
print("Output directory: ", out_perevent, "exists:", out_perevent.exists())


# quick smoke test (replace with an event folder that exists in your clone)
test_event = "2005ap"
print((supernovae_dir / test_event / f"{test_event}.txt").exists())


Repo root:         /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric


NameError: name 'supernovae_dir' is not defined

In [31]:


# --- Toggle env like your example ---
Cristina = True
Shar = not Cristina

if Cristina:
    print("[CONFIG] Using Cristina's local MacBook setup")
    sys.path.insert(0, "/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub")
    os.environ["RUBIN_SIM_DATA_DIR"] = "/Users/andradenebula/rubin_sim_data"
    db_dir   = "/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub"
    base_dir = "/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub/output"
else:
    print("[CONFIG] Using Shar's Dirac server setup")
    sys.path.insert(0, "/lustre/lrspec/metrics")
    sys.path.insert(0, "/home/3155/metrics/Multi_Transient_Metrics_Hub")
    os.environ["RUBIN_SIM_DATA_DIR"] = "/lustre/lrspec/metrics/rubin_sim_data"
    db_dir   = "/lustre/lrspec/metrics"
    base_dir = "/lustre/lrspec/metrics/results"



[CONFIG] Using Cristina's local MacBook setup


In [ ]:
# --- Modules ---
sys.path.append(os.path.abspath(".."))
s_u = "shared_utils"
slsn_metric_mod = "SLSNe.slsn_metric"
maker_mod       = "SLSNe.make_slsn_templates"

# fresh reload
for m in [s_u, slsn_metric_mod, maker_mod]:
    if m in sys.modules: del sys.modules[m]
shared_utils   = __import__(s_u);            importlib.reload(shared_utils)
slsn_metric    = __import__(slsn_metric_mod, fromlist=["*"]); importlib.reload(slsn_metric)
slsn_templates = __import__(maker_mod, fromlist=["*"]);       importlib.reload(slsn_templates)

print("[INFO] Loaded shared_utils, slsn_metric, make_slsn_templates")

# --- File locations from your SLSN export step ---
# (1) Per-event photometry CSVs created by export_slsne_photometry.py
per_event_csv_dir = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/Rubin_tests")        # e.g., output/photometry/per_event
# (2) Optional all-parameters table (CSV) to get 'redshift_med', 'Peak_MJD_med', etc.
allparams_csv     = Path("/path/to/all_parameters.csv")         # or None if not available

######## you left off trying to figure out if you should use the text file or create a csv of sebastian's all params file. make a csv in all_run then use here. 

# --- Build a single templates.pkl (only needs to run when new catalog is added) ---
templates_file = Path(base_dir) / "SLSNe" / "SLSNe_catalog_templates.pkl"
slsn_templates.build_slsn_templates(
    per_event_dir     = per_event_csv_dir,
    allparams_csv     = allparams_csv,
    out_templates_pkl = templates_file,
    n_time            = 220,
    t_post_days       = 160.0,
    min_points_for_fit= 8,
    force_restframe   = True,
    make_abs_mag      = True
)

# --- Load templates into an LC model (GRB-compatible API) ---
shared_lc_model = shared_utils.load_or_generate_templates(
    slsn_metric.LC, templates_file=str(templates_file), generate_new=False
)

# (Optional) quick visual check
shared_utils.plot_template_lcs(str(templates_file), num=3, use_log_time=True, plot_overlap=True, ylim=( -24, -16 ))

# --- Population & cadences (your usual knobs) ---
use_kcorrect=True
k_correct_type='powerlaw'
k_correct_arg=-0.75

testname=None
testname_metric_only=None

generate_new_pop = True
make_debug_plots = True

rate_density = 1e-8   # placeholder; you’ll set this later
z_min, z_max = 0.02, 2.0
d_min, d_max = None, None
gal_lat_cut  = None
use_extinction = True
t_start, t_end = 1, 3652

cadences = ['baseline_v5.0.0_10yrs', 'four_roll_v5.0.0_10yrs']
ignore_triples = True
clean_temp = True

# --- Build output paths with your helper ---
templates_file_, pop_file, df_file, storage_dir, summary_filename = shared_utils.build_filenames(
    rate_density=rate_density, z_min=z_min, z_max=z_max, d_min=d_min, d_max=d_max,
    science_case="SLSNe", testname=testname, testname_metric_only=testname_metric_only,
    ignore_triples=ignore_triples, use_extinction=use_extinction, use_kcorrect=use_kcorrect,
    base_dir=base_dir
)
# We already have a templates_file above; we’ll keep using that.


In [ ]:
#load and/or generate light curves
shared_lc_model = shared_utils.load_or_generate_templates(metric.LC,
    templates_file=templates_file,
    generate_new=generate_new_templates, num_lightcurves=num_lightcurves
)

#plot light curves from pkl file if desired
shared_utils.plot_template_lcs(templates_file, num=3, use_log_time=False, plot_overlap=True)

# Not yet

In [ ]:

# --- Population slicer (reuses your GRB machinery; templates are absolute-mag) ---
slicer = shared_utils.load_or_generate_population(
    use_extinction=use_extinction,
    lc_model=shared_lc_model,
    t_start=t_start, t_end=t_end,
    d_min=d_min, d_max=d_max, z_min=z_min, z_max=z_max,
    seed=42, num_lightcurves=None,  # read count from templates; not used
    gal_lat_cut=gal_lat_cut, rate_density=rate_density,
    pop_file=pop_file,
    generate_new=generate_new_pop, make_debug_plots=make_debug_plots,
    use_kcorrect=use_kcorrect, k_correct_type=k_correct_type, k_correct_arg=k_correct_arg
)

# --- Run metrics (detect placeholder here; add more later if desired) ---
multi_metrics = slsn_metric.get_multi_metrics(shared_lc_model, include=['detect'],
                                              use_extinction=use_extinction,
                                              use_kcorrect=use_kcorrect,
                                              k_correct_type=k_correct_type,
                                              k_correct_arg=k_correct_arg)

# optional: diagnostics knobs (same as your GRB code)
for m in multi_metrics:
    if hasattr(m, "diag_store"):
        m.diag_store = False
        m.diag_sample_rate = 0.01
        m.diag_per_event_cap = 30
        m.diag_min_snr = 3
        m.diag_max_mag = None

# quick detect run (produces ObsRecords_*.csv like GRB flow)
df_obs = shared_utils.run_detect(
    slsn_metric, slicer, cadences, shared_lc_model, db_dir, storage_dir, df_file,
    use_extinction=use_extinction, use_kcorrect=use_kcorrect,
    k_correct_type=k_correct_type, k_correct_arg=k_correct_arg,
    ignore_triples=ignore_triples, debug=True, plot=True, clean_temp=clean_temp
)

# multi-metric summary table (works even with only 'detect' present)
summary_df = shared_utils.run_multi_metrics(
    multi_metrics, slicer, cadences, shared_lc_model,
    db_dir, storage_dir, summary_filename=summary_filename,
    ignore_triples=False, plot=True, clean_temp=clean_temp, use_extinction=use_extinction
)
summary_df
